In [3]:
%reload_ext autoreload
%autoreload 2

# get the dataset first

In [4]:
import sys
sys.path.append('/rhome/sawale/indus_traning/mlm-fine-tuning/mlm')

In [18]:
import json
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from datasets import DatasetDict, Dataset

from preprocess_data import preprocess_dataset

config_path = "../config_new_data.json"
with open(config_path, "r") as file:
    config = json.load(file)

data_src = "local"
n_rows = None
lm_dataset, tokenizer, data_collator = preprocess_dataset(
        config.get("input"),
        data_src,
        n_rows,
    )
lm_dataset

Map (num_proc=4): 100%|██████████| 9745/9745 [00:01<00:00, 5436.25 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 98566
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 12404
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 12283
    })
})

In [19]:
# Initialize the embedding model
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

def compute_token_importance(examples):
    token_ids = examples["input_ids"]
    
    # Decode the full sentence
    detokenized_text = tokenizer.decode(token_ids, skip_special_tokens=True)
    
    # Decode each individual token
    tokens_decoded = tokenizer.convert_ids_to_tokens(token_ids)

    # Compute embeddings
    sentence_embedding = emb_model.encode(detokenized_text, convert_to_tensor=True)
    token_embeddings = torch.stack([
        emb_model.encode(token, convert_to_tensor=True) for token in tokens_decoded
    ])

    # Compute cosine similarity
    cosine_similarities = torch.nn.functional.cosine_similarity(sentence_embedding, token_embeddings, dim=1)
    
    # Convert to list
    return {"token_imp": cosine_similarities.tolist()}

# Apply function to each dataset split
lm_dataset = lm_dataset.map(compute_token_importance)


Map:   0%|          | 26/98566 [00:53<56:25:04,  2.06s/ examples]


KeyboardInterrupt: 

hashing inputs: 
- location of input data
- embedding model
- nrows

In [12]:

# Check available GPUs
device = "cuda" if torch.cuda.is_available() else "cpu"

# Use DataParallel if multiple GPUs are available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    emb_model = SentenceTransformer("all-MiniLM-L6-v2")
    emb_model = torch.nn.DataParallel(emb_model)  # Wrap in DataParallel
else:
    emb_model = SentenceTransformer("all-MiniLM-L6-v2")

# Move model to GPU
emb_model = emb_model.to(device)


def compute_token_importance(examples):
    token_ids = examples["input_ids"]
    
    # Decode the full sentence
    detokenized_text = tokenizer.decode(token_ids, skip_special_tokens=True)
    
    # Decode individual tokens
    tokens_decoded = tokenizer.convert_ids_to_tokens(token_ids)

    # Compute embeddings in batches for efficiency
    with torch.no_grad():
        sentence_embedding = emb_model.module.encode([detokenized_text], convert_to_tensor=True, device=device)
        
        # Encode all tokens at once for parallelism
        token_embeddings = emb_model.module.encode(tokens_decoded, convert_to_tensor=True, device=device)
    
    # Compute cosine similarity
    cosine_similarities = torch.nn.functional.cosine_similarity(sentence_embedding, token_embeddings, dim=1)
    
    return {"token_imp": cosine_similarities.cpu().tolist()}

# Apply function to dataset in parallel
lm_dataset = lm_dataset.map(compute_token_importance, batched=True)


Using 4 GPUs!


Map:   0%|          | 0/16 [00:00<?, ? examples/s]


TypeError: argument 'ids': 'list' object cannot be interpreted as an integer

In [ ]:
from datasets import Dataset, DatasetDict, load_dataset
from typing import Any, Dict, List, Optional, Tuple, Union

n_rows = 1000
ds = DatasetDict(load_dataset(
    **{
        "path": "parquet",
        "data_files": "/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/data/parsed_sde_init_check_with_title_filtered_word_count.parquet"
    }
))

if n_rows:
        ds = DatasetDict(
            {split: ds[split].select(range(n_rows)) for split in ds.keys()},
        )


def compute_text_level_embeddings(batch: Dict[str, List[str]], text_column: str, model: SentenceTransformer) -> Dict[str, List[List[float]]]:
    """Computes text embeddings for a batch of text data."""
    embeddings = model.encode(batch[text_column], convert_to_numpy=True)  # Encode text
    batch["text_embedding"] = [emb.tolist() for emb in embeddings]  # Convert to list for Dataset compatibility
    return batch

def generate_text_level_embeddings(ds: DatasetDict, text_column: str) -> DatasetDict:
    """Generates and stores text embeddings in the dataset."""
    emb_model = SentenceTransformer("all-MiniLM-L6-v2")
    emb_model = emb_model.to("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available
    
    ds = ds.map(
        compute_text_level_embeddings,
        batched=True,
        batch_size=32,
        fn_kwargs={"model": emb_model, "text_column": text_column},
    )
    return ds


ds = generate_sentence_level_embeddings(ds, "text")

Map: 100%|██████████| 1000/1000 [00:04<00:00, 207.00 examples/s]


In [35]:
ds

DatasetDict({
    train: Dataset({
        features: ['url_1', 'first_field', 'second_field', 'third_field', 'fourth_field', 'old_text', 'normalized_url', 'collection__config_folder', 'url_2', 'generated_title', 'scraped_title', 'division_display', 'document_type_display', 'line', 'word_counts', 'text', 'sentence_embedding'],
        num_rows: 1000
    })
})